### Chapter 2.2 - Data Preprocessing

Raw data rarely arrives as clean tensors. It often starts as a table, a CSV file, or a spreadsheet-like object with missing values, text categories, and columns that must be separated into inputs and labels.

This chapter builds a small preprocessing pipeline in layers: plain Python containers, pandas tables, NumPy arrays, and finally PyTorch tensors.


In [ ]:
import numpy as np
import pandas as pd
import torch
from io import StringIO


### 2.2.1 Reading the Dataset

#### 1. Intuition

A dataset is a collection of examples. An example is one item the model can learn from, such as one house, one customer, or one image.

For table data, each row usually represents one example. Each column usually represents one variable. A feature is an input variable used to make a prediction. A label is the target value we want to predict.

A CSV file is a plain-text table where values are separated by commas. CSV stands for comma-separated values.

#### 2. Why this exists

Most real ML projects begin by loading data from disk. Before tensors appear, we need to read the raw file and inspect the rows, columns, and missing values.

If this step is wrong, every later tensor can be numerically valid but semantically wrong. Semantic means related to meaning. A tensor can have the right shape but still contain the wrong column in the wrong place.


#### 3. Examples

First, read tiny table data manually from a string. This is not how large projects work, but it makes the structure visible.


In [ ]:
csv_text = "name,rooms,price\nA,2,100\nB,3,150"
lines = csv_text.split("\n")
header = lines[0].split(",")
rows = [line.split(",") for line in lines[1:]]
header, rows


Now use pandas. A pandas DataFrame is a table object with named columns and indexed rows.


In [ ]:
csv_text = "name,rooms,price\nA,2,100\nB,3,150"
data = pd.read_csv(StringIO(csv_text))
data


A missing value is a place where the dataset does not provide a value. Pandas commonly displays missing values as `NaN`, which means not a number.


In [ ]:
csv_text = "name,rooms,price\nA,2,100\nB,,150"
data = pd.read_csv(StringIO(csv_text))
data.isna()


#### 4. Step-by-step breakdown

`csv_text` stores a tiny CSV as ordinary text.

`split("\n")` separates the text into lines.

`split(",")` separates each line into fields.

`pd.read_csv(StringIO(csv_text))` asks pandas to parse CSV-formatted text as a table. `StringIO` makes a string behave like a small file object.

`data.isna()` returns a table of Boolean values. A Boolean value is either `True` or `False`. Here, `True` means the original cell is missing.

#### 5. Connection to ML systems

Data loading usually happens before model code. In a larger system, this stage may read CSV files, SQL tables, Parquet files, image folders, or web datasets.

The important ML idea is that raw storage format and model input format are different. Reading the dataset is the bridge between them.

#### 6. Common confusion points

- A DataFrame is not a tensor. It is a table object used for data work.
- A row is usually one example, but this is a convention, not a law.
- A column can be a feature, a label, metadata, or something to ignore.
- `NaN` means missing or invalid numeric data, not the string `"NaN"`.


### 2.2.2 Data Preparation

#### 1. Intuition

Data preparation means changing raw data into a cleaner form before tensor conversion.

Two common problems appear immediately. First, some values are missing. Second, some values are categories written as text, such as `red`, `blue`, or `green`.

A categorical variable is a variable whose value belongs to a set of named groups. A numerical variable is a variable whose value is already a number.

#### 2. Why this exists

Models compute with numbers. They cannot directly multiply by the word `blue` or compare a blank cell mathematically.

Missing numerical values often need imputation. Imputation means filling in a missing value with a chosen replacement, such as the column mean.

Categorical values often need encoding. Encoding means representing non-numeric information with numbers.


#### 3. Examples

Plain Python version: fill a missing number with the average of available values.


In [ ]:
values = [2, None, 4]
known = [v for v in values if v is not None]
mean_value = sum(known) / len(known)
filled = [mean_value if v is None else v for v in values]
filled


Pandas version: fill a missing numerical column with the column mean.


In [ ]:
data = pd.DataFrame({"rooms": [2, None, 4]})
mean_rooms = data["rooms"].mean()
data["rooms"] = data["rooms"].fillna(mean_rooms)
data


One-hot encoding represents each category with separate indicator columns. An indicator is a 0 or 1 value showing whether a condition is true.


In [ ]:
data = pd.DataFrame({"color": ["red", "blue", "red"]})
encoded = pd.get_dummies(data, dummy_na=False)
encoded


Prepare a tiny table with both numerical and categorical columns.


In [ ]:
data = pd.DataFrame({
    "rooms": [2, None, 4],
    "color": ["red", "blue", "red"]
})
data["rooms"] = data["rooms"].fillna(data["rooms"].mean())
prepared = pd.get_dummies(data)
prepared


#### 4. Step-by-step breakdown

The manual Python example first removes `None` values from the average calculation.

`mean_value` stores the replacement number.

The list comprehension creates a new list. If a value is `None`, it inserts the mean. Otherwise, it keeps the original value.

`data["rooms"].mean()` computes the average of the non-missing values in the column.

`fillna(mean_rooms)` replaces missing entries with the chosen value.

`pd.get_dummies(data)` creates one column per category. For example, `color_red` is 1 when the original color was red and 0 otherwise.

#### 5. Connection to ML systems

Data preparation determines what numerical information the model receives. A careless encoding choice can create misleading inputs.

For example, coding `red=1`, `blue=2`, `green=3` may accidentally tell the model that green is larger than red. One-hot encoding avoids that fake ordering.

#### 6. Common confusion points

- Filling missing values is a modeling choice, not a neutral action.
- One-hot encoding increases the number of columns.
- Text categories must be encoded before tensor conversion.
- `None` is a Python missing object; `NaN` is a numerical missing marker used by libraries like pandas and NumPy.


### 2.2.3 Conversion to the Tensor Format

#### 1. Intuition

After data is cleaned, we convert it into tensors because PyTorch models operate on tensors.

The input tensor usually contains features. The label tensor usually contains target values. Keeping inputs and labels separate makes the training task explicit.

#### 2. Why this exists

A DataFrame is convenient for cleaning, but it is not the main computation object for deep learning.

A tensor has a numerical dtype, a shape, and device information. Dtype means data type, such as floating-point number or integer. Device means where the tensor is stored, such as CPU or GPU.


#### 3. Examples

Start with a prepared table and split it into features and labels.


In [ ]:
data = pd.DataFrame({
    "rooms": [2.0, 3.0, 4.0],
    "price": [100.0, 150.0, 210.0]
})
features = data[["rooms"]]
labels = data["price"]
features, labels


Convert pandas objects into NumPy arrays, then into PyTorch tensors.


In [ ]:
X_np = features.to_numpy()
y_np = labels.to_numpy()
X = torch.tensor(X_np, dtype=torch.float32)
y = torch.tensor(y_np, dtype=torch.float32)
X.shape, y.shape


A label vector can be reshaped into a column when a model expects one target column per row.


In [ ]:
y_column = y.reshape(-1, 1)
y_column.shape


#### 4. Step-by-step breakdown

`data[["rooms"]]` uses double brackets so the result stays a DataFrame with one column.

`data["price"]` uses single brackets, so the result is a pandas Series. A Series is like one labeled column.

`to_numpy()` converts pandas data into NumPy arrays.

`torch.tensor(..., dtype=torch.float32)` creates a tensor using 32-bit floating-point numbers. Floating-point numbers can represent decimals.

`reshape(-1, 1)` means: infer the number of rows automatically and use 1 column. The `-1` asks PyTorch to calculate that dimension from the number of values.

#### 5. Connection to ML systems

Most supervised learning code expects something like `X` for inputs and `y` for labels.

Supervised learning means learning from examples where the correct answer is provided. The model sees input features and learns to predict the label.

#### 6. Common confusion points

- Strings cannot be directly converted into useful model tensors.
- Features and labels should usually be separated before training.
- `float32` is common in deep learning because it balances precision and memory use.
- Shape `(3,)` and `(3, 1)` are different even though both contain three values.


### 2.2.4 Discussion

#### 1. Intuition

A preprocessing pipeline is an ordered sequence of data changes. Each step should have a clear reason.

The basic flow is: read raw data, inspect it, handle missing values, encode categories, split features and labels, then convert to tensors.

#### 2. Why this exists

Preprocessing decisions can change model behavior as much as the model architecture itself.

One important risk is data leakage. Data leakage happens when information from the target or future data accidentally enters the training inputs. This makes evaluation look better than reality.


#### 3. Examples

A tiny pipeline can be written as explicit steps so each transformation is visible.


In [ ]:
raw = pd.DataFrame({"x": [1, None, 3], "y": [2, 4, 6]})
clean = raw.copy()
clean["x"] = clean["x"].fillna(clean["x"].mean())
X = torch.tensor(clean[["x"]].to_numpy(), dtype=torch.float32)
y = torch.tensor(clean["y"].to_numpy(), dtype=torch.float32)
X, y


#### 4. Step-by-step breakdown

`raw.copy()` creates a separate DataFrame so the original raw data is preserved.

The missing `x` value is filled using the mean of the `x` column.

The feature tensor uses only the `x` column.

The label tensor uses the `y` column.

The code is short, but it represents the same pattern used in larger projects.

#### 5. Connection to ML systems

Training systems often hide preprocessing inside dataset classes or data loaders. A dataset class is a Python object that knows how to return one example. A data loader is an object that repeatedly asks the dataset for examples and groups them into batches.

Those abstractions are easier to understand when the raw preprocessing steps are clear first.

#### 6. Common confusion points

- Preprocessing is part of the model pipeline, not separate from it.
- Cleaning choices should be reproducible.
- Never use test labels to choose preprocessing values.
- Inspect shapes after conversion, not only before conversion.


### 2.2.5 Exercises

#### 1. Intuition

The goal of these exercises is to practice changing messy table data into tensors deliberately.

#### 2. Why this exists

Preprocessing bugs often appear as silent mistakes. Exercises help you build the habit of checking each intermediate object.


#### 3. Examples

Exercise 1: Read a tiny CSV string into a DataFrame.


In [ ]:
csv_text = "size,color,price\n1,red,10\n,blue,12\n3,red,18"
data = pd.read_csv(StringIO(csv_text))
data


Exercise 2: Fill the missing numerical value with the column mean.


In [ ]:
data["size"] = data["size"].fillna(data["size"].mean())
data


Exercise 3: One-hot encode the color column and convert features to a tensor.


In [ ]:
prepared = pd.get_dummies(data)
X_np = prepared.drop(columns=["price"]).astype(float).to_numpy()
X = torch.tensor(X_np, dtype=torch.float32)
X.shape


Exercise 4: Convert the label column to a tensor.


In [ ]:
y = torch.tensor(prepared["price"].to_numpy(), dtype=torch.float32)
y


#### 4. Step-by-step breakdown

Exercise 1 checks whether the CSV rows and columns were read correctly.

Exercise 2 checks numerical missing-value imputation.

Exercise 3 checks categorical encoding and feature tensor conversion.

Exercise 4 checks label tensor conversion.

#### 5. Connection to ML systems

These are the same operations used before training a small tabular regression model. Regression means predicting a numerical value, such as price.

#### 6. Common confusion points

- `drop(columns=["price"])` removes the label from the input features.
- One-hot encoding changes column names and count.
- Tensor conversion should happen after text columns are encoded.
- Always check `X.shape` and `y.shape` before training.
